# Smart Parking Detection — Notebook Workflow

Notebook ini menjalankan alur proyek UAS dari dataset PKLot/Roboflow berformat COCO:

1. Memeriksa struktur dan kategori dataset.
2. Memfilter gambar berdasarkan waktu **12:00:00 sampai 15:00:59** dari timestamp pada nama file.
3. Menyalin anotasi COCO yang cocok dengan gambar terpilih.
4. Mengonversi COCO ke format YOLO.
5. Melatih YOLO11 untuk dua kelas: `space-empty` dan `space-occupied`.
6. Mengevaluasi model dan menyiapkan aplikasi Streamlit.

**Catatan metodologi:** metadata COCO yang sudah diperiksa belum memuat kamera dan cuaca. Notebook ini hanya memfilter waktu. Jangan menyatakan satu angle dan cuaca cerah sudah diterapkan sebelum sumber kamera/cuaca berhasil diverifikasi.

**Cara pakai:** simpan notebook ini di folder utama `smart_parkir_detection`, sejajar dengan folder `dataset`. Jalankan sel dari atas ke bawah.


## 1. Persiapan lokasi proyek dan library

In [57]:
from pathlib import Path
import json
import re
import shutil
from datetime import time
from collections import Counter

# Notebook diasumsikan berada di folder utama project.


INPUT_DIR = Path('/kaggle/input/datasets/ammarnassanalhajali/pklot-dataset')
WORK_DIR = Path('/kaggle/working/')

RAW_DIR = INPUT_DIR  # langsung ke train / valid / test
FILTERED_DIR = WORK_DIR / "dataset" / "filtered"
YOLO_DIR = WORK_DIR / "dataset" / "yolo"
MODEL_DIR = WORK_DIR / "models"

SPLITS = ["train", "valid", "test"]
START_TIME = time(12, 0, 0)
END_TIME = time(15, 0, 59)
print("Dataset raw  :", RAW_DIR)
print("Raw tersedia :", RAW_DIR.exists())

if not RAW_DIR.exists():
    raise FileNotFoundError(
        "Folder dataset/raw tidak ditemukan. Pastikan notebook berada di "
        "folder utama smart_parkir_detection dan struktur dataset/raw/train, "
        "dataset/raw/valid, dataset/raw/test sudah benar."
    )


Dataset raw  : /kaggle/input/datasets/ammarnassanalhajali/pklot-dataset
Raw tersedia : True


In [18]:
# Install dependensi jika belum terpasang.
%pip install -q ultralytics opencv-python-headless pillow numpy PyYAML streamlit


Note: you may need to restart the kernel to use updated packages.


## 2. Periksa dataset COCO

In [48]:
ANNOTATION_NAMES = [
    "_annotations.coco.json",
    "_annotations.coco",
    "_annotations.json",
]

def find_annotation(folder: Path) -> Path:
    for name in ANNOTATION_NAMES:
        p = folder / name
        if p.exists():
            return p
    raise FileNotFoundError(f"Anotasi COCO tidak ditemukan di {folder}")

for split in SPLITS:
    folder = RAW_DIR / split
    ann_path = find_annotation(folder)
    coco = json.loads(ann_path.read_text(encoding="utf-8"))
    print(f"\\n{split.upper()}")
    print("File anotasi :", ann_path.name)
    print("Jumlah gambar:", len(coco.get("images", [])))
    print("Jumlah anotasi:", len(coco.get("annotations", [])))
    print("Kategori:", [c["name"] for c in coco.get("categories", [])])
    print("Contoh nama file:", coco["images"][0]["file_name"] if coco.get("images") else "-")


\nTRAIN
File anotasi : _annotations.coco.json
Jumlah gambar: 8691
Jumlah anotasi: 497856
Kategori: ['spaces', 'space-empty', 'space-occupied']
Contoh nama file: 2013-04-10_12_00_06_jpg.rf.bef78f7879666a6d8997ce513fd8d124.jpg
\nVALID
File anotasi : _annotations.coco.json
Jumlah gambar: 2483
Jumlah anotasi: 143316
Kategori: ['spaces', 'space-empty', 'space-occupied']
Contoh nama file: 2012-12-21_18_10_14_jpg.rf.62e89081eb0427cd3a372345b23ed409.jpg
\nTEST
File anotasi : _annotations.coco.json
Jumlah gambar: 1242
Jumlah anotasi: 70684
Kategori: ['spaces', 'space-empty', 'space-occupied']
Contoh nama file: 2012-11-06_13_43_31_jpg.rf.00207754dbce0d625d24ca31898fcfd7.jpg


In [49]:
from pathlib import Path

folder = Path('/kaggle/input/datasets/ammarnassanalhajali/pklot-dataset')
print('Isi folder train:')
for p in sorted(folder.iterdir())[:20]:
  print(' -', p.name, '(dir)' if p.is_dir() else '')

print('\nSemua file .json di bawah folder train:')
for j in folder.rglob('*.json'):
  print(' -', j)

Isi folder train:
 - README.dataset.txt 
 - README.roboflow.txt 
 - test (dir)
 - train (dir)
 - valid (dir)

Semua file .json di bawah folder train:
 - /kaggle/input/datasets/ammarnassanalhajali/pklot-dataset/test/_annotations.coco.json
 - /kaggle/input/datasets/ammarnassanalhajali/pklot-dataset/train/_annotations.coco.json
 - /kaggle/input/datasets/ammarnassanalhajali/pklot-dataset/valid/_annotations.coco.json


## 3. Filter gambar berdasarkan waktu

Timestamp diambil dari nama file, misalnya:

`2013-04-16_10_50_05_jpg.rf....jpg`

Notebook memilih waktu **12:00:00–15:00:59**, termasuk seluruh menit pada pukul 15.00.

Folder `dataset/raw` tidak diubah. Hasil filter disimpan ke `dataset/filtered`.

Jika nama file tidak mengandung timestamp yang dikenali, gambar dilewati dan dihitung pada laporan.


In [50]:
from datetime import time
import re

TIMESTAMP_PATTERN = re.compile(
    r"(?P<date>\d{4}-\d{2}-\d{2})_(?P<hour>\d{2})_(?P<minute>\d{2})_(?P<second>\d{2})"
)


def get_capture_time(filename: str):
  match = TIMESTAMP_PATTERN.search(filename)
  if not match:
    return None
  try:
    return time(
        int(match.group("hour")),
        int(match.group("minute")),
        int(match.group("second")),
    )
  except ValueError:
    return None

def locate_image(split_dir: Path, file_name: str):
    p = split_dir / file_name
    if p.exists():
        return p
    p = split_dir / Path(file_name).name
    if p.exists():
        return p
    return None

# Hati-hati: folder hasil filter sebelumnya akan dibuat ulang.
if FILTERED_DIR.exists():
    shutil.rmtree(FILTERED_DIR)
FILTERED_DIR.mkdir(parents=True, exist_ok=True)

filter_summary = {}

for split in SPLITS:
    split_dir = RAW_DIR / split
    coco = json.loads(find_annotation(split_dir).read_text(encoding="utf-8"))

    selected_images = []
    selected_ids = set()
    skipped_time = 0
    skipped_timestamp = 0
    missing_image = 0

    for image in coco.get("images", []):
        filename = image.get("file_name", "")
        capture_time = get_capture_time(filename)

        if capture_time is None:
            skipped_timestamp += 1
            continue

        if not (START_TIME <= capture_time <= END_TIME):
            skipped_time += 1
            continue

        source_image = locate_image(split_dir, filename)
        if source_image is None:
            missing_image += 1
            print(f"[GAMBAR TIDAK DITEMUKAN] {split}/{filename}")
            continue

        selected_images.append(image)
        selected_ids.add(image["id"])

        destination = FILTERED_DIR / split / Path(filename).name
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source_image, destination)

    selected_annotations = [
        ann for ann in coco.get("annotations", [])
        if ann.get("image_id") in selected_ids
    ]

    filtered_coco = {
        "info": coco.get("info", {}),
        "licenses": coco.get("licenses", []),
        "categories": coco.get("categories", []),
        "images": [
            {**image, "file_name": Path(image["file_name"]).name}
            for image in selected_images
        ],
        "annotations": selected_annotations,
    }

    output_json = FILTERED_DIR / split / "_annotations.coco.json"
    output_json.parent.mkdir(parents=True, exist_ok=True)
    output_json.write_text(json.dumps(filtered_coco, indent=2), encoding="utf-8")

    filter_summary[split] = {
        "selected_images": len(selected_images),
        "selected_annotations": len(selected_annotations),
        "skipped_outside_time": skipped_time,
        "skipped_unreadable_timestamp": skipped_timestamp,
        "missing_image_files": missing_image,
    }

print("FILTERING SELESAI")
for split, result in filter_summary.items():
    print(split.upper(), result)
print("Total gambar terpilih:", sum(x["selected_images"] for x in filter_summary.values()))
print("Hasil disimpan di:", FILTERED_DIR)


FILTERING SELESAI
TRAIN {'selected_images': 2166, 'selected_annotations': 122996, 'skipped_outside_time': 6525, 'skipped_unreadable_timestamp': 0, 'missing_image_files': 0}
VALID {'selected_images': 607, 'selected_annotations': 33504, 'skipped_outside_time': 1876, 'skipped_unreadable_timestamp': 0, 'missing_image_files': 0}
TEST {'selected_images': 303, 'selected_annotations': 17340, 'skipped_outside_time': 939, 'skipped_unreadable_timestamp': 0, 'missing_image_files': 0}
Total gambar terpilih: 3076
Hasil disimpan di: /kaggle/working/dataset/filtered


### 3A. Periksa hasil filtering sebelum lanjut

Jangan lanjut jika folder `valid` atau `test` kosong. Periksa juga apakah jumlah gambar masuk akal. Filtering ini hanya berdasarkan jam, bukan angle kamera atau cuaca.


In [51]:
import json
from pathlib import Path
import re

for split in SPLITS:
  coco = json.loads(
      find_annotation(RAW_DIR / split).read_text(encoding='utf-8')
  )
  names = [img.get('file_name', '') for img in coco.get('images', [])[:5]]
  hours = []
  for img in coco.get('images', []):
    t = get_capture_time(img.get('file_name', ''))
    if t:
      hours.append(t.hour)
  print(f'=== {split.upper()} ===')
  print('Contoh file_name:', names)
  print('Distribusi jam terbaca:', sorted(set(hours)))

=== TRAIN ===
Contoh file_name: ['2013-04-10_12_00_06_jpg.rf.bef78f7879666a6d8997ce513fd8d124.jpg', '2012-10-28_15_07_11_jpg.rf.bf0215bf1be731828c9fda8d04a0a12b.jpg', '2012-10-15_15_16_04_jpg.rf.bf069ee1dcc96c9e5c2c816b6d079f5c.jpg', '2013-03-20_11_50_06_jpg.rf.bf0daeaae52ff23b8ccce43ca836241c.jpg', '2012-12-24_09_50_05_jpg.rf.bf1406d0cd01a8d266dd21f669e2b9f4.jpg']
Distribusi jam terbaca: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
=== VALID ===
Contoh file_name: ['2012-12-21_18_10_14_jpg.rf.62e89081eb0427cd3a372345b23ed409.jpg', '2013-03-20_07_50_02_jpg.rf.62ea5a91da3e88ec2d1c819dae6549c7.jpg', '2012-09-12_12_16_56_jpg.rf.63035c9fd47dc1422cfab65d6659dcf3.jpg', '2012-09-17_14_24_11_jpg.rf.63304c351176227663633dc099299cc3.jpg', '2013-01-21_17_25_13_jpg.rf.6355eab55ae421d87bddd45f49f52dcb.jpg']
Distribusi jam terbaca: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21]
=== TEST ===
Contoh file_name: ['2012-11-06_13_43_31_jpg.rf.00207754dbce0d625d24ca31898f

In [52]:
for split in SPLITS:
    folder = FILTERED_DIR / split
    images = list(folder.glob("*.jpg")) + list(folder.glob("*.jpeg")) + list(folder.glob("*.png"))
    coco = json.loads(find_annotation(folder).read_text(encoding="utf-8"))
    print(f"{split}: {len(images)} file gambar | {len(coco.get('annotations', []))} anotasi")

if any(filter_summary[s]["selected_images"] == 0 for s in SPLITS):
    raise RuntimeError(
        "Setidaknya satu split kosong setelah filtering. Periksa dataset dan timestamp "
        "sebelum melanjutkan konversi/training."
    )


train: 2166 file gambar | 122996 anotasi
valid: 607 file gambar | 33504 anotasi
test: 303 file gambar | 17340 anotasi


## 4. Konversi COCO ke YOLO

Kita menggunakan dua kelas yang benar-benar relevan untuk tugas:

- class 0: `space-empty`
- class 1: `space-occupied`

Kategori induk `spaces` tidak dilatih sebagai kelas deteksi.


In [55]:
import yaml

if YOLO_DIR.exists():
    shutil.rmtree(YOLO_DIR)

TARGET_CLASSES = ["space-empty", "space-occupied"]
SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}

for source_split, yolo_split in SPLIT_MAP.items():
    folder = FILTERED_DIR / source_split
    coco = json.loads(find_annotation(folder).read_text(encoding="utf-8"))

    category_by_name = {c["name"]: c["id"] for c in coco.get("categories", [])}
    missing = [name for name in TARGET_CLASSES if name not in category_by_name]
    if missing:
        raise ValueError(f"Kategori tidak ditemukan di {source_split}: {missing}")

    category_to_yolo = {
        category_by_name["space-empty"]: 0,
        category_by_name["space-occupied"]: 1,
    }

    annotations_by_image = {}
    for ann in coco.get("annotations", []):
        if ann.get("category_id") in category_to_yolo:
            annotations_by_image.setdefault(ann["image_id"], []).append(ann)

    image_out = YOLO_DIR / "images" / yolo_split
    label_out = YOLO_DIR / "labels" / yolo_split
    image_out.mkdir(parents=True, exist_ok=True)
    label_out.mkdir(parents=True, exist_ok=True)

    converted = 0
    for image in coco.get("images", []):
        filename = Path(image["file_name"]).name
        source_image = folder / filename
        if not source_image.exists():
            raise FileNotFoundError(f"Gambar tidak ditemukan: {source_image}")

        shutil.copy2(source_image, image_out / filename)
        width = float(image["width"])
        height = float(image["height"])
        lines = []

        for ann in annotations_by_image.get(image["id"], []):
            x, y, box_w, box_h = map(float, ann["bbox"])
            if box_w <= 0 or box_h <= 0:
                continue

            class_id = category_to_yolo[ann["category_id"]]
            center_x = (x + box_w / 2) / width
            center_y = (y + box_h / 2) / height
            norm_w = box_w / width
            norm_h = box_h / height

            lines.append(
                f"{class_id} {center_x:.6f} {center_y:.6f} "
                f"{norm_w:.6f} {norm_h:.6f}"
            )

        (label_out / f"{Path(filename).stem}.txt").write_text(
            "\n".join(lines), encoding="utf-8"
        )
        converted += 1

    print(f"{source_split}: {converted} gambar dikonversi")

data_yaml = {
    "path": str(YOLO_DIR.resolve()).replace("\\\\", "/"),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {0: "space-empty", 1: "space-occupied"},
}
(YOLO_DIR / "data.yaml").write_text(
    yaml.safe_dump(data_yaml, sort_keys=False),
    encoding="utf-8"
)
print("data.yaml dibuat:", YOLO_DIR / "data.yaml")


train: 2166 gambar dikonversi
valid: 607 gambar dikonversi
test: 303 gambar dikonversi
data.yaml dibuat: /kaggle/working/dataset/yolo/data.yaml


## 5. Training YOLO11

In [58]:
from ultralytics import YOLO

DATA_YAML = YOLO_DIR / "data.yaml"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# YOLO11 nano cocok untuk percobaan awal dan relatif ringan.
model = YOLO("yolo11n.pt")

train_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=8,
    workers=2,
    project=str(MODEL_DIR),
    name="smart_parking",
    patience=12,
    plots=True,
    exist_ok=True,
)

best_weight = MODEL_DIR / "smart_parking" / "weights" / "best.pt"
if best_weight.exists():
    shutil.copy2(best_weight, MODEL_DIR / "best.pt")
    print("Model terbaik disimpan:", MODEL_DIR / "best.pt")
else:
    print("best.pt belum ditemukan. Periksa output training Ultralytics.")


Ultralytics 8.4.219 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset/yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=smart_parking, nbs=64, nms=None, opset=None, optimiz

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      2.44G      1.218     0.7302      1.023        540        640: 100% ━━━━━━━━━━━━ 271/271 10.7it/s 25.4s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 6.3it/s 6.1s0.2s
                   all        607      33504      0.939      0.965      0.981      0.693

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/50      2.44G      1.098     0.6419     0.9722        490        640: 100% ━━━━━━━━━━━━ 271/271 10.7it/s 25.4s0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 6.3it/s 6.0s0.2s
                   all        607      33504      0.944       0.97      0.975      0.719

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       4/50      2.44G      1.032     0.5948     0.9493        389        640: 100% ━━━━━━━━━━━━ 271/271 10.6it/s 25.6s0.2ss
                 Class     

## 6. Evaluasi pada test set

In [59]:
best_weight = MODEL_DIR / "best.pt"
if not best_weight.exists():
    raise FileNotFoundError("models/best.pt tidak ditemukan. Pastikan training berhasil.")

best_model = YOLO(str(best_weight))
metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    batch=8,
)

print("Evaluasi test selesai.")
print("mAP50:", metrics.box.map50)
print("mAP50-95:", metrics.box.map)


Ultralytics 8.4.219 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO11n summary (fused): 101 layers, 2,597,294 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1747.4±657.3 MB/s, size: 66.3 KB)
val: Scanning /kaggle/working/dataset/yolo/labels/test... 303 images, 9 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 303/303 1.1Kit/s 0.3s<0.2s
val: New cache created: /kaggle/working/dataset/yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 38/38 8.8it/s 4.3s<0.2s
                   all        303      17340      0.989      0.979      0.988      0.959
           space-empty        252       9741      0.995       0.96      0.989      0.963
        space-occupied        256       7599      0.983      0.999      0.987      0.954
Speed: 1.3ms preprocess, 5.1ms inference, 0.0ms loss, 1.2ms postprocess per image
Results saved to /kaggle/working/runs/detect/val
Evaluas

## 7. Buat aplikasi Streamlit (`app.py`)

Sel ini membuat file `app.py` di folder proyek. Aplikasi menampilkan gambar asli dan hasil deteksi, serta menghitung jumlah kotak slot yang terdeteksi untuk masing-masing kelas.


In [ ]:
app_source = r"""
from pathlib import Path
import io
import cv2
import numpy as np
import streamlit as st
from PIL import Image
from ultralytics import YOLO

ROOT = Path(__file__).resolve().parent
MODEL_PATH = ROOT / "models" / "best.pt"

st.set_page_config(page_title="Smart Parking Detection", page_icon="🅿️", layout="wide")
st.title("🅿️ Smart Parking Detection")
st.caption("Deteksi slot parkir kosong dan terisi menggunakan YOLO11.")

@st.cache_resource
def load_model():
    return YOLO(str(MODEL_PATH))

def detect(image, model, confidence):
    rgb = np.array(image.convert("RGB"))
    result = model.predict(source=rgb, conf=confidence, verbose=False)[0]
    annotated = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

    empty = 0
    occupied = 0
    details = []

    for box in result.boxes:
        class_id = int(box.cls[0].item())
        label = str(result.names[class_id]).lower().replace("-", "_")
        score = float(box.conf[0].item())
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        if label == "space_empty":
            empty += 1
            status = "KOSONG"
            color = (0, 190, 0)
        elif label == "space_occupied":
            occupied += 1
            status = "TERISI"
            color = (0, 0, 230)
        else:
            continue

        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 2)
        cv2.putText(annotated, f"{status} {score:.2f}",
                    (x1, max(20, y1 - 7)), cv2.FONT_HERSHEY_SIMPLEX,
                    0.5, color, 2)
        details.append({"Status": status, "Confidence": round(score, 3)})

    output = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    return output, empty, occupied, details

if not MODEL_PATH.exists():
    st.error("Model models/best.pt belum tersedia. Jalankan notebook training terlebih dahulu.")
    st.stop()

confidence = st.slider("Confidence threshold", 0.05, 0.95, 0.25, 0.05)
uploaded = st.file_uploader("Unggah gambar area parkir", type=["jpg", "jpeg", "png"])

if uploaded:
    original = Image.open(uploaded).convert("RGB")
    with st.spinner("Mendeteksi slot parkir..."):
        output, empty, occupied, details = detect(original, load_model(), confidence)

    left, right = st.columns(2)
    left.subheader("Gambar asli")
    left.image(original, use_container_width=True)
    right.subheader("Hasil deteksi")
    right.image(output, use_container_width=True)

    total = empty + occupied
    st.subheader("Ringkasan ketersediaan parkir")
    c1, c2, c3 = st.columns(3)
    c1.metric("Total slot terdeteksi", total)
    c2.metric("Terisi", occupied)
    c3.metric("Kosong", empty)

    if total:
        st.progress(occupied / total)
        st.caption(f"Okupansi: {occupied / total * 100:.1f}%")
    else:
        st.warning("Tidak ada slot yang terdeteksi. Coba gambar lain atau periksa model.")

    if details:
        st.subheader("Detail deteksi")
        st.dataframe(details, use_container_width=True, hide_index=True)

    buffer = io.BytesIO()
    Image.fromarray(output).save(buffer, format="PNG")
    st.download_button("Download hasil deteksi", buffer.getvalue(),
                       "smart_parking_result.png", "image/png")
else:
    st.info("Unggah gambar parkir untuk memulai.")
"""

(ROOT / "app.py").write_text(app_source, encoding="utf-8")
print("app.py berhasil dibuat:", ROOT / "app.py")


## 8. Jalankan website Streamlit

In [61]:
# Jalankan perintah ini dari terminal VS Code, bukan sebagai sel Python biasa:
# streamlit run app.py
#
# Jika ingin menjalankannya dari notebook:
import subprocess, sys
print("Jalankan di terminal: streamlit run app.py")


Jalankan di terminal: streamlit run app.py


## 9. Batasan dan catatan laporan

- Filtering notebook ini hanya memakai jam dari nama file, yaitu 12:00:00–15:00:59.
- JSON yang diperiksa sebelumnya tidak memperlihatkan metadata angle kamera atau cuaca.
- Karena itu, hasil belum boleh disebut khusus satu angle dan cuaca cerah.
- Model menghitung jumlah slot berdasarkan objek `space-empty` dan `space-occupied` yang terdeteksi. Angka hasil deteksi adalah prediksi model, bukan jumlah slot yang dijamin benar.
- Pertahankan dataset mentah dan jangan mengunggah dataset besar ke GitHub.
